In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [5]:
df = pd.read_csv('Dataset/alert_earthquakes_cleaned_with_null.csv')

In [6]:
df.dtypes

magnitude      float64
alert              str
tsunami          int64
magType            str
longitude      float64
latitude       float64
depth          float64
country            str
subnational        str
distanceKM     float64
month            int64
day              int64
hour_sin       float64
hour_cos       float64
dtype: object

In [7]:
import torch

print("--- Teste de Hardware PyTorch ---")
print(f"Versão do PyTorch: {torch.__version__}")

# Verifica se o CUDA (GPU) está disponível
cuda_disponivel = torch.cuda.is_available()
print(f"CUDA disponível? {cuda_disponivel}")

if cuda_disponivel:
    # Mostra o nome da placa gráfica detetada
    nome_gpu = torch.cuda.get_device_name(0)
    print(f"Placa Gráfica detetada: {nome_gpu}")
    print("\n✅ Tudo perfeito! Pode avançar para o treino do modelo.")
else:
    print("\n❌ A GPU não foi detectada. O PyTorch está configurado para usar apenas a CPU.")

--- Teste de Hardware PyTorch ---
Versão do PyTorch: 2.11.0+cu130
CUDA disponível? True
Placa Gráfica detetada: NVIDIA GeForce RTX 5080

✅ Tudo perfeito! Pode avançar para o treino do modelo.


# Treinamento

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        """
        alpha: Um tensor com os pesos de cada classe (para desbalanceamento).
        gamma: O fator de foco (2.0 é o padrão ouro na indústria).
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # 1. Calcula o Cross Entropy normal, mas sem fazer a média final ainda ('none')
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # 2. Extrai a probabilidade da classe correta (pt)
        # Como o Cross Entropy no PyTorch é matematicamente igual a -log(pt),
        # podemos reverter isso aplicando a exponencial (exp)
        pt = torch.exp(-ce_loss)

        # 3. Aplica o fator Focal Loss: (1 - pt)^gamma * CE
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        # 4. Se tivermos os pesos de classe (Alpha), aplicamos agora
        if self.alpha is not None:
            # Puxa o peso específico para cada sismo do lote atual
            alpha_t = self.alpha.gather(0, targets.data.view(-1))
            focal_loss = alpha_t * focal_loss

        # 5. Retorna a média de erro do lote (batch) inteiro
        return focal_loss.mean()

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------------------------------------------------
# 1. PRÉ-PROCESSAMENTO (Idêntico ao Keras, usando Scikit-Learn)
# ---------------------------------------------------------
#df = pd.read_csv("Earthquake_Alert_Model/Dataset/alert_earthquakes_cleaned_with_null.csv")

df = pd.read_csv('Dataset/alert_earthquakes_cleaned_with_null.csv')

features_categoricas = ['country', 'subnational']
features_numericas = ['magnitude','tsunami',]# 'depth', 'latitude', 'longitude', 'distanceKM', 'day', 'month', 'hour_sin', 'hour_cos', 'is_night', 'is_shallow',  'destruction_index', 'energy_release']

X = df[features_categoricas + features_numericas]
y = df['alert']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_categoricas)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=50135, stratify=y_encoded)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# ---------------------------------------------------------
# 2. CONVERSÃO PARA TENSORES DO PYTORCH
# ---------------------------------------------------------
# O PyTorch exige que os dados sejam convertidos para os seus próprios tipos de tensores
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # Classes devem ser do tipo Long (inteiros)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Criar os DataLoaders para alimentar a rede em lotes (batches) de 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ---------------------------------------------------------
# 3. DEFINIÇÃO DA ARQUITETURA DA REDE NEURAL (Orientação a Objetos)
# ---------------------------------------------------------
class EarthquakeNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(EarthquakeNet, self).__init__()

        # 1ª Camada: Recebe as suas features e joga para 256 neurônios (era 128)
        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256) # O BatchNorm DEVE ter o mesmo número da camada!
        self.drop1 = nn.Dropout(0.3)

        # 2ª Camada: Recebe os 256 de cima e afunila para 128 neurônios
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.drop2 = nn.Dropout(0.3)

        # 3ª Camada (NOVA!): Recebe os 128 e afunila para 64
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.drop3 = nn.Dropout(0.3)

        # Camada de Saída: Recebe os 64 da última camada oculta e cospe as 5 cores de alerta
        self.out = nn.Linear(64, num_classes)

        self.relu = nn.GELU()

    def forward(self, x):
        # Aqui é onde você define o "caminho" que os dados vão percorrer.
        # Precisamos adicionar a nova linha para a informação passar pela camada 3!
        x = self.drop1(self.relu(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu(self.bn2(self.fc2(x))))
        x = self.drop3(self.relu(self.bn3(self.fc3(x)))) # <- A nova camada entra aqui
        x = self.out(x)

        #x = self.drop1(self.relu(self.fc1(x)))
        #x = self.drop2(self.relu(self.fc2(x)))
        #x = self.drop3(self.relu(self.fc3(x))) # <- A nova camada entra aqui
        #x = self.out(x)

        return x

# # mais camadas
# class EarthquakeNet_Deep(nn.Module):
#     def __init__(self, input_size, num_classes):
#         super(EarthquakeNet_Deep, self).__init__()
#
#         # 1ª Camada (Ainda mais larga: 512)
#         self.fc1 = nn.Linear(input_size, 512)
#         self.bn1 = nn.BatchNorm1d(512)
#         self.drop1 = nn.Dropout(0.4) # Aumentamos o Dropout para tentar conter a "decoreba"
#
#         # 2ª Camada (256)
#         self.fc2 = nn.Linear(512, 256)
#         self.bn2 = nn.BatchNorm1d(256)
#         self.drop2 = nn.Dropout(0.4)
#
#         # 3ª Camada (128)
#         self.fc3 = nn.Linear(256, 128)
#         self.bn3 = nn.BatchNorm1d(128)
#         self.drop3 = nn.Dropout(0.3)
#
#         # 4ª Camada (NOVA! - 64)
#         self.fc4 = nn.Linear(128, 64)
#         self.bn4 = nn.BatchNorm1d(64)
#         self.drop4 = nn.Dropout(0.3)
#
#         # Camada de Saída
#         self.out = nn.Linear(64, num_classes)
#
#         self.relu = nn.ReLU()
#
#     def forward(self, x):
#         # O caminho agora é muito mais longo
#         x = self.drop1(self.relu(self.bn1(self.fc1(x))))
#         x = self.drop2(self.relu(self.bn2(self.fc2(x))))
#         x = self.drop3(self.relu(self.bn3(self.fc3(x))))
#         x = self.drop4(self.relu(self.bn4(self.fc4(x)))) # <- A 4ª camada passa por aqui
#         x = self.out(x)
#         return x


input_shape = X_train_processed.shape[1]
model = EarthquakeNet(input_size=input_shape, num_classes=num_classes)

# ---------------------------------------------------------
# 4. CONFIGURAÇÃO DE TREINO (Loss e Otimizador)
# ---------------------------------------------------------
# Mover o modelo para a GPU se disponível (NVIDIA CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 1. Calcula matematicamente o peso justo para cada cor de alerta
pesos_classes = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# 2. Converte para o formato que a placa gráfica entende
pesos_tensor = torch.tensor(pesos_classes, dtype=torch.float32).to(device)

# 3. Injeta os pesos na função de erro do PyTorch
#criterion = nn.CrossEntropyLoss(weight=pesos_tensor)
criterion = FocalLoss(alpha=pesos_tensor, gamma=1.5)


optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# Reduz a taxa de aprendizado pela metade (factor=0.5) se não houver melhora em 5 rodadas (patience=5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# ---------------------------------------------------------
# 5. O LOOP DE TREINAMENTO MANUAL (Com Early Stopping)
# ---------------------------------------------------------
epochs = 10000
patience = 1000

# Inicializar variáveis do Early Stopping
best_val_f1 = 0.0
patience_counter = 0

print(f"A iniciar o treino no dispositivo: {device.type.upper()}\n")

for epoch in range(epochs):
    # --- MODO DE TREINO ---
    model.train()
    running_loss = 0.0

    # Listas para guardar o que aconteceu no treino
    train_preds = []
    train_targets = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()       # 1. Zerar os gradientes da rodada anterior
        outputs = model(inputs)     # 2. Forward pass (Previsão)
        loss = criterion(outputs, labels) # 3. Calcular o Erro
        loss.backward()             # 4. Backpropagation (Calcular a culpa de cada peso)
        optimizer.step()            # 5. Atualizar os pesos

        running_loss += loss.item()

        # Extrair previsões da rodada de treino
        _, predicted = torch.max(outputs.data, 1)
        train_preds.extend(predicted.cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    avg_train_loss = running_loss / len(train_loader)

    # --- MODO DE AVALIAÇÃO (Validação) ---
    model.eval()
    val_loss = 0.0

    # Listas para guardar o que aconteceu na validação
    val_preds = []
    val_targets = []

    with torch.no_grad(): # Desliga o cálculo de gradientes para poupar memória na validação
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            # Extrair previsões da rodada de validação
            _, predicted = torch.max(outputs.data, 1)
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(test_loader)

    # ==========================================
    # 3. CÁLCULO DAS MÉTRICAS (Usando Scikit-Learn)
    # ==========================================
    # zero_division=0 evita avisos vermelhos se o modelo falhar em prever alguma cor no início

    # Métricas de Validação (O mais importante)
    val_acc = accuracy_score(val_targets, val_preds)
    val_prec = precision_score(val_targets, val_preds, average='macro', zero_division=0)
    val_rec = recall_score(val_targets, val_preds, average='macro', zero_division=0)
    val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)

    # ==========================================
    # 4. IMPRESSÃO DO PAINEL DE CONTROLO
    # ==========================================
    print(f"Epoch [{epoch+1:03d}/{epochs:03d}] "
          f"| T_Loss: {avg_train_loss:.4f} | V_Loss: {avg_val_loss:.4f} "
          f"|| Acc: {val_acc:.3f} | Prec: {val_prec:.3f} | Rec: {val_rec:.3f} | F1: {val_f1:.3f}")

    # (Opcional)
    # O agendador analisa o erro de validação e decide se pisa no freio
    scheduler.step(avg_val_loss)

    # ==========================================
    # 5. LÓGICA DE EARLY STOPPING (Focada no F1)
    # ==========================================
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        best_model_weights = model.state_dict()
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\n🛑 Early Stopping ativado na Epoch {epoch+1}.")
        print(f"Restaurando os pesos do melhor F1-Score: {best_val_f1:.4f}")
        model.load_state_dict(best_model_weights)
        break

print("\n✅ Treino PyTorch Concluído!")
print(f"\nMelhor Epoch: {epoch+1-patience}.")
print(f"Restaurando os pesos do melhor F1-Score: {best_val_f1:.4f}")

A iniciar o treino no dispositivo: CUDA

Epoch [001/10000] | T_Loss: 1.4089 | V_Loss: 0.9988 || Acc: 0.315 | Prec: 0.286 | Rec: 0.323 | F1: 0.198
Epoch [002/10000] | T_Loss: 1.1801 | V_Loss: 0.9231 || Acc: 0.651 | Prec: 0.296 | Rec: 0.437 | F1: 0.282
Epoch [003/10000] | T_Loss: 1.0966 | V_Loss: 0.9141 || Acc: 0.647 | Prec: 0.307 | Rec: 0.488 | F1: 0.293
Epoch [004/10000] | T_Loss: 0.9225 | V_Loss: 0.8688 || Acc: 0.589 | Prec: 0.291 | Rec: 0.463 | F1: 0.269
Epoch [005/10000] | T_Loss: 0.9307 | V_Loss: 0.8441 || Acc: 0.606 | Prec: 0.296 | Rec: 0.448 | F1: 0.276
Epoch [006/10000] | T_Loss: 0.8448 | V_Loss: 0.8528 || Acc: 0.607 | Prec: 0.293 | Rec: 0.456 | F1: 0.273
Epoch [007/10000] | T_Loss: 0.8551 | V_Loss: 0.8398 || Acc: 0.681 | Prec: 0.313 | Rec: 0.469 | F1: 0.302
Epoch [008/10000] | T_Loss: 0.8710 | V_Loss: 0.7842 || Acc: 0.630 | Prec: 0.297 | Rec: 0.465 | F1: 0.282
Epoch [009/10000] | T_Loss: 0.7473 | V_Loss: 0.8204 || Acc: 0.703 | Prec: 0.313 | Rec: 0.408 | F1: 0.303
Epoch [010/100

In [10]:
# Definir o nome do ficheiro (a extensão padrão do PyTorch é .pth)
caminho_modelo = "modelo_terremotos.pth"

# Salvar o conhecimento do modelo
torch.save(model.state_dict(), caminho_modelo)
print(f"🧠 Pesos da Rede Neural salvos com sucesso em: {caminho_modelo}")

🧠 Pesos da Rede Neural salvos com sucesso em: modelo_terremotos.pth


In [ ]:
import joblib

# Salvar as ferramentas de transformação
joblib.dump(preprocessor, 'preprocessor_terremotos.pkl')
joblib.dump(label_encoder, 'label_encoder_terremotos.pkl')
print("🛠️ Transformadores salvos com sucesso!")

In [ ]:
import torch
import joblib

# 1. Carregar as ferramentas de pré-processamento
preprocessor = joblib.load('preprocessor_terremotos.pkl')
label_encoder = joblib.load('label_encoder_terremotos.pkl')

# 2. Recriar a arquitetura "vazia" da rede
# (O Python precisa saber o formato da rede antes de injetar os pesos.
# As variáveis input_shape e num_classes devem ser as mesmas do momento do treino)
modelo_carregado = EarthquakeNet(input_size=input_shape, num_classes=num_classes)

# 3. Injetar o conhecimento (os pesos) na rede vazia
modelo_carregado.load_state_dict(torch.load('modelo_terremotos.pth'))

# 4. Colocar o modelo em modo de avaliação (Crucial!)
# Isto desliga o Dropout e avisa a rede que ela já não está a treinar, mas sim a fazer previsões reais.
modelo_carregado.eval()

print("✅ Sistema completamente carregado e pronto para prever novos alertas!")